# GoFlow — Agent décisionnel à partir des prévisions XGBoost

Le forecasting répond : **« Que va-t-il se passer dans 15 minutes ? »**
L'agent répond : **« Que recommander en fonction de cette prévision et du contexte ? »**

Le prérequis XGBoost est désormais disponible. Ce notebook remplace le diagnostic
bloqué par un pipeline exécutable, sans réentraînement ni modification des benchmarks.


## 1. Contrat et environnement

Vérifier les capacités locales sans installer de dépendances. Si aucun petit LLM
instruct local n'est disponible, utiliser l'agent déterministe et conserver une
interface remplaçable. Les sorties finales sont un JSON strict de cinq champs ;
la source de décision est conservée séparément dans une enveloppe de suivi.


In [1]:
# Utiliser la même implémentation que les agents, depuis la racine ou notebooks/.
from pathlib import Path
import copy
import json
import sys

import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/agents/decision_agent.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.agents.decision_agent import create_runtime

# Isoler les données, les seuils et le backend des autres agents du même processus.
_rules = create_runtime()
TRAFFIC_STATES = _rules.TRAFFIC_STATES
RISK_LEVELS = _rules.RISK_LEVELS
ACTIONS = _rules.ACTIONS
DECISION_KEYS = _rules.DECISION_KEYS
CONFIG = _rules.CONFIG
MAPPING_GROUPS = _rules.MAPPING_GROUPS
SYSTEM_PROMPT = _rules.SYSTEM_PROMPT
file_hash = _rules.file_hash
inspect_environment = _rules.inspect_environment


## 2. Charger et réaligner les 455 090 prévisions

Vérifier toutes les clés location/timestep, les valeurs finies, la provenance du CSV
brut et la concordance des mesures et cibles. L'audit lit les cibles futures pour
contrôler l'export ; le state builder ne les reçoit jamais.

Le graphe 170×170 symétrisé provient des distances réelles ASTGCN PEMS08.
Le mapping reste **une hypothèse pour 25 capteurs aux séries identiques** ;
leur identité physique ne peut pas être prouvée par leurs valeurs seules.


In [2]:
def initialize(root):
    # Exposer les données pour les explications et tests, sans recopier la logique.
    result = _rules.initialize(root)
    for name in ("ROOT", "PREDICTIONS", "OBSERVATIONS", "ADJACENCY", "LOOKUP",
                 "THRESHOLDS", "ENVIRONMENT"):
        globals()[name] = getattr(_rules, name)
    return result


In [3]:
# Résoudre la racine depuis notebooks/ ou depuis le projet.
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'notebooks/03_model_benchmark.ipynb').exists())
display(initialize(root))
print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2))
print('Seuils expérimentaux ajustés sur train uniquement :')
print(json.dumps(THRESHOLDS, ensure_ascii=False, indent=2))


Prévisions validées : (455090, 11) ; graphe : (170, 170)


,timestep,location,current_flow,current_speed,current_occupancy,target_occupy_t15,day_index,slot_in_day,hour,minute,predicted_occupancy_t15
0,15177,0,465.0,62.3,0.1569,0.1547,52.0,200.0,16.0,40.0,0.154230
1,15178,0,499.0,64.3,0.1562,0.1472,52.0,201.0,16.0,45.0,0.151665
2,15179,0,500.0,65.3,0.1532,0.1467,52.0,202.0,16.0,50.0,0.152489
3,15180,0,471.0,64.3,0.1547,0.1537,52.0,203.0,16.0,55.0,0.155828
4,15181,0,454.0,65.4,0.1472,0.1472,52.0,204.0,17.0,0.0,0.158575


{
  "python": "3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]",
  "packages": {
    "torch": true,
    "transformers": true,
    "llama_cpp": false,
    "ollama": false,
    "psutil": true
  },
  "ollama_executable": null,
  "gguf_files": [],
  "torch_version": "2.11.0+cpu",
  "cuda_available": false,
  "ram_total_bytes": 16487878656,
  "ram_available_bytes": 3629977600,
  "ollama_models": [],
  "ollama_status": "URLError",
  "selected_llm": null,
  "llm_mode": "placeholder_rule_based"
}
Seuils expérimentaux ajustés sur train uniquement :
{
  "occupancy_median": 0.0599,
  "occupancy_high": 0.1071,
  "occupancy_severe": 0.1604,
  "speed_low": 62.6,
  "speed_median": 64.9,
  "strong_change": 0.015000000000000006
}


## 3. Construire un état causal et son contexte spatial

Les tendances utilisent t−6…t−1 du même capteur : différence entre les extrémités,
divisée par cinq intervalles. Les voisins sont les indices connectés par une arête
réelle, avec observations et prévisions au même t. Aucune relation n'est déduite
de numéros consécutifs. Incidents et urgences restent `null`.


In [4]:
# Construction causale et récupération des voisins dans le module partagé.
build_traffic_state = _rules.build_traffic_state
indicators = _rules.indicators


## 4. Règles et validation JSON

Les quantiles train sont des seuils opérationnels **expérimentaux**. La politique
prend en compte l'occupation courante et prévue, la hausse prévue, la vitesse, les
tendances et les voisins. Elle ne connaît aucun plan de feux : aucune extension
ou réduction de phase n'est autorisée dans ce contexte. `confidence=0.5` est
non calibrée et ne représente pas une probabilité scientifique.

La validation refuse aussi les champs supplémentaires, les clés JSON dupliquées,
NaN, Infinity et un booléen utilisé comme confidence. Le contrat de grounding
vérifie ensuite la cohérence et l'ancrage de la justification aux données.


In [5]:
# Contrat, règles et garde-fous partagés avec le code de production.
rule_based_decision = _rules.rule_based_decision
strict_json_loads = _rules.strict_json_loads
validate_decision = _rules.validate_decision
decision_contract = _rules.decision_contract
is_grounded_decision = _rules.is_grounded_decision


## 5. Interface LLM, consigne française et fallback

Le system prompt est défini dans la première cellule de code. Le LLM reçoit les
valeurs réelles, les indicateurs et les choix autorisés. Cette version lui permet
de choisir une justification parmi deux textes calculés ; elle refuse toute
justification libre susceptible d'introduire des faits non fournis.

`final_decision` vérifie l'entrée contre les sources, tente le LLM, puis utilise
les règles si le LLM est absent, échoue ou produit une sortie invalide/non ancrée.
L'appel reçoit une copie de l'état pour empêcher la modification des entrées.


In [6]:
# L'interface et le fallback utilisent le même contexte que le state builder.
llm_decision = _rules.llm_decision
final_decision = _rules.final_decision


## 6. Sélection automatique de cinq vrais cas test

Sélectionner des observations distinctes par critères sur les mesures courantes,
les prévisions et les voisins. Aucun critère ne consulte la cible réellement
observée à t+3. Ce sont des illustrations extrêmes, pas un échantillon représentatif.


In [7]:
# Sélection déterministe conservée, sans réentraîner ni recalculer de prévisions.
select_scenarios = _rules.select_scenarios


In [8]:
# Afficher les critères et les valeurs mesurées/prédites réellement sélectionnées.
scenarios = select_scenarios()
display(scenarios[['scenario', 'location', 'timestep', 'current_occupancy',
                   'predicted_occupancy_t15', 'occupancy_change', 'current_speed',
                   'neighbor_occupancy_range', 'selection_criterion']])
assert len(scenarios) == 5
assert not scenarios.duplicated(['location', 'timestep']).any()


,scenario,location,timestep,current_occupancy,predicted_occupancy_t15,occupancy_change,current_speed,neighbor_occupancy_range,selection_criterion
0,trafic_fluide,33,15227,0.0000,0.002740,0.002740,65.0,0.0237,current_occupancy
1,occupation_elevee,23,17200,0.7237,0.346324,-0.377376,5.9,0.0153,current_occupancy
2,forte_hausse_predite,134,16252,0.0030,0.270631,0.267631,35.9,0.1154,occupancy_change
3,amelioration_prevue,17,15331,0.6938,0.198641,-0.495159,44.2,0.0590,occupancy_change
4,voisins_contrastes,34,17530,0.0458,0.043239,-0.002561,67.2,0.6866,neighbor_occupancy_range


## 7. Décisions et grille de validation

Chaque cas affiche l'entrée complète avec ses voisins, la décision stricte, sa
source `llm` ou `rule_based` et la grille. Les résultats décrivent la conformité
au contrat, pas une accuracy LLM ni l'effet sur la congestion réelle.


In [9]:
# Helpers expérimentaux explicites : aucune sauvegarde au simple import du module.
evaluate_scenarios = _rules.evaluate_scenarios
save_results = _rules.save_results


In [10]:
# Évaluer les cinq états réels avec l'interface actuellement disponible.
inputs, outputs, validations = evaluate_scenarios(scenarios)
for input_record, output_record in zip(inputs, outputs):
    print('\nSCÉNARIO :', input_record['scenario'])
    print(json.dumps(input_record['state'], ensure_ascii=False, indent=2, allow_nan=False))
    print(json.dumps(output_record, ensure_ascii=False, indent=2, allow_nan=False))
display(validations)



SCÉNARIO : trafic_fluide
{
  "sensor_id": 33,
  "timestep": 15227,
  "current_flow": 0.0,
  "current_speed": 65.0,
  "current_occupancy": 0.0,
  "predicted_occupancy_t15": 0.0027397831,
  "occupancy_change": 0.0027397831,
  "recent_occupancy_trend": 0.0001,
  "recent_speed_trend": 0.019999999999998862,
  "neighbors": [
    {
      "sensor_id": 32,
      "timestep": 15227,
      "current_speed": 61.8,
      "current_occupancy": 0.0381,
      "predicted_occupancy_t15": 0.039587665
    },
    {
      "sensor_id": 156,
      "timestep": 15227,
      "current_speed": 64.5,
      "current_occupancy": 0.0144,
      "predicted_occupancy_t15": 0.013298589
    }
  ],
  "incident_detected": null,
  "emergency_vehicle_present": null
}
{
  "scenario": "trafic_fluide",
  "decision": {
    "traffic_state": "free_flow",
    "risk_level": "low",
    "recommended_action": "keep_current_signal_plan",
    "reason": "Occupation actuelle=0.000000, prévue à +15 min=0.002740, variation=+0.002740; vitesse=65.

,scenario,source,json_valid,authorized_action,canonical_input_verified,reason_grounded_in_input,missing_information_signaled,risk_action_coherent,no_fabricated_data_in_contract,high_occupancy_neighbors
0,trafic_fluide,rule_based,True,True,True,True,True,True,True,0
1,occupation_elevee,rule_based,True,True,True,True,True,True,True,0
2,forte_hausse_predite,rule_based,True,True,True,True,True,True,True,1
3,amelioration_prevue,rule_based,True,True,True,True,True,True,True,1
4,voisins_contrastes,rule_based,True,True,True,True,True,True,True,1


## 8. Tests unitaires de garde-fous

Les entrées de trafic proviennent d'un scénario réel. Les sorties LLM malformées
ou contradictoires ci-dessous sont **des fixtures de test**, pas des observations
ni des décisions de modèle enregistrées. Un appel valide simulé teste le chemin
LLM ; cela ne signifie pas qu'un modèle instruct a été exécuté.


In [11]:
# Vérifier le contrat et les branches de fallback sur un état réel.
state = inputs[0]['state']
baseline = rule_based_decision(state)
assert validate_decision(json.dumps(baseline, allow_nan=False))
assert not validate_decision('{')
assert not validate_decision({**baseline, 'confidence': True})
assert not validate_decision({**baseline, 'confidence': float('nan')})
assert not validate_decision({**baseline, 'confidence': 1.1})
assert not validate_decision({**baseline, 'recommended_action': 'invented_action'})
assert not validate_decision({**baseline, 'reason': ' '})
assert not validate_decision({**baseline, 'extra': 1})
duplicated = json.dumps(baseline)[:-1] + ', "confidence": 0.5}'
assert not validate_decision(duplicated)
assert not is_grounded_decision({**baseline, 'reason': 'Accident détecté.'}, state)
assert not is_grounded_decision({**baseline, 'recommended_action': 'extend_green_phase'}, state)

# Vérifier les tendances exactement sur le passé et le voisinage sur les arêtes réelles.
sensor, timestep = state['sensor_id'], state['timestep']
history = OBSERVATIONS[sensor, timestep - 7:timestep - 1]
assert state['recent_occupancy_trend'] == float((history[-1, 2] - history[0, 2]) / 5)
assert state['recent_speed_trend'] == float((history[-1, 1] - history[0, 1]) / 5)
assert {n['sensor_id'] for n in state['neighbors']} == set(np.flatnonzero(ADJACENCY[sensor] > 0)) - {sensor}
assert all(n['timestep'] == timestep for n in state['neighbors'])
assert state['incident_detected'] is None and state['emergency_vehicle_present'] is None

# Tester l'indisponibilité, l'erreur, le JSON invalide, la fabrication et un retour valide simulé.
original_llm = _rules.llm_decision
try:
    for fixture in [None, '{', {**baseline, 'reason': 'Accident détecté.'}]:
        _rules.llm_decision = lambda input_state, answer=fixture: answer
        result = final_decision(state)
        assert result['source'] == 'rule_based' and result['decision'] == baseline
    def failed_llm(input_state):
        raise OSError('Erreur simulée uniquement pour le test')
    _rules.llm_decision = failed_llm
    assert final_decision(state)['source'] == 'rule_based'
    _rules.llm_decision = lambda input_state: json.dumps(baseline, allow_nan=False)
    assert final_decision(state)['source'] == 'llm'
finally:
    _rules.llm_decision = original_llm
tampered = copy.deepcopy(state)
tampered['predicted_occupancy_t15'] += 1
try:
    final_decision(tampered)
except ValueError:
    pass
else:
    raise AssertionError('Une entrée modifiée doit être refusée')
print('Tests du schéma, du grounding, des tendances, des voisins et du fallback : OK.')


Tests du schéma, du grounding, des tendances, des voisins et du fallback : OK.


## 9. Sauvegarde des résultats réels

Sauvegarder les cinq scénarios, états, décisions et validations sous
`results/decision_agent/`, ainsi qu'une méthode précisant les seuils, l'environnement
et les empreintes des sources. Le CSV de benchmark n'est jamais écrit.


In [12]:
# Documenter les choix et limites avec les valeurs réelles de cette exécution.
methodology = "# GoFlow — Agent Traffic Prediction & Adaptive Signal Control\n\nLe forecasting répond : « Que va-t-il se passer dans 15 minutes ? »\nL'agent répond : « Que recommander en fonction de cette prévision et du contexte ? »\n\nL'agent réutilise exclusivement le CSV XGBoost sauvegardé : 455 090 observations\ntest, cible target_occupy_t15, horizon +3 timesteps de 5 minutes. Aucun modèle n'est\nréentraîné et aucune nouvelle prévision n'est calculée. Le contrôle initial compare\nles clés, les mesures courantes et les cibles au CSV brut. Les cibles futures ne\nsont jamais incluses dans les états ni utilisées pour sélectionner les scénarios.\n\n## Temporalité et seuils expérimentaux\n\nLes tendances sont (valeur(t−1) − valeur(t−6))/5, dans les unités source par timestep.\nLes six mesures appartiennent au même capteur. Elles excluent t et tout futur.\nLes seuils sont calculés seulement sur train, timesteps 13–12497 : occupancy aux\nquantiles 50 %, 90 %, 97 %, speed au quantile 25 % et médiane, variation absolue\nd'occupancy sur trois pas au quantile 90 %. La cible la plus tardive de ce calcul\nest 12500, avant la validation commençant à 12501. Ces seuils sont des choix\nopérationnels expérimentaux, pas des classes de congestion scientifiquement validées.\nAucun seuil n'est ajusté sur validation ou test. Le test sert ici à des exemples\ndécisionnels illustratifs, pas à une estimation de performance de contrôle.\n\nLe niveau utilise le maximum de l'occupation courante et prévue. Une occupation\nau quantile 97 % entraîne severe/high ; au quantile 90 % ou avec hausse forte,\ncongested/high. Au-dessus de la médiane, à faible vitesse, avec un voisin à occupation\nélevée ou avec hausse passée\net baisse passée de vitesse conjointes, moderate/medium ; sinon free_flow/low.\nPour high, l'action est coordinate_with_neighbor si au moins un voisin est au\nquantile élevé courant, sinon manual_review. Medium entraîne monitor ; low,\nkeep_current_signal_plan. Une amélioration prévue peut donc conserver un risque\nélevé si l'occupation courante est encore élevée. La confidence constante 0.5\nest un indicateur neutre non calibré, pas une probabilité statistique de réussite.\n\n## Graphe et mapping\n\nL'adjacence STGCN sauvegardée est une matrice symétrisée 170×170, construite à partir\ndu fichier de distances ASTGCN PEMS08, avec 274 liens non dirigés. Sa connectivité\nest comparée aux arêtes du fichier source. On conserve toutes les arêtes réellement\nprésentes, même avec un poids gaussien très petit, sans seuil arbitraire. Les voisins\nsont directs et leur état ainsi que leur prévision sont lus au même timestep.\nL'adjacence porte sur le réseau de capteurs, pas sur des feux de carrefours.\n\nLe mapping location=index NPZ reste une hypothèse pour 25 capteurs aux séries\nentièrement identiques : [48,61,80,121,122,123,141,142],\n[50,51,74,75,90,91,115,131], [55,70,73,81,83,84,159], [62,109].\nLes valeurs seules ne prouvent pas leur identité physique individuelle. Les conseils\nde coordination restent donc conditionnels à ce mapping documenté.\n\n## LLM et garde-fous\n\nL'environnement est vérifié avant toute installation : Python, RAM, GPU, packages,\nOllama local, transformers et llama.cpp. Aucun téléchargement ni installation.\nChronos n'est pas un modèle instruct décisionnel. Un petit modèle Ollama déjà local,\nidentifié comme instruct, peut être utilisé si la mémoire mesurée suffit ; sinon\nllm_decision retourne None et le fallback fonctionne. L'interface est remplaçable.\nLa présence de transformers seule ne prouve pas la présence d'un modèle instruct.\n\nLa première version borne volontairement le LLM : les règles imposent état, risque\net confidence ; le LLM peut choisir une action parmi les actions compatibles et une\njustification parmi deux formulations exactes dérivées des données. Une justification\nlibre est refusée, même avec un JSON valide. Ce contrôle permet de vérifier l'ancrage\naux valeurs ; il ne valide pas l'efficacité réelle de la politique de trafic.\nSans information de feux, extend_green_phase et reduce_green_phase sont dans le\nschéma général mais interdites par le contrat applicable. Aucune durée de feu\nn'est proposée et aucune commande n'est envoyée à un système externe.\n\nincident_detected et emergency_vehicle_present sont null, jamais False inventé.\nChaque justification signale aussi le plan de feux indisponible. À terme, Incident\nDetection Agent et Emergency Agent fourniront ces champs ; Cooperative Traffic\nAgent fournira le contexte spatial, Traffic Prediction Agent les prévisions, et\nl'Orchestrator combinera ces informations. Ce notebook implémente seulement le\nrôle Traffic Prediction & Adaptive Signal Control.\n\n## Scénarios et validation\n\nCinq cas distincts sont sélectionnés par critères déterministes sur le test :\noccupation minimale parmi les états fluides, occupation maximale parmi les états\nélevés, hausse prévue maximale, baisse prévue maximale et contraste maximal entre\nvoisins. Il s'agit de cas illustratifs extrêmes, pas d'un échantillon représentatif.\nLes états, prévisions et voisins proviennent des données ; aucune valeur n'est\nécrite manuellement pour construire un scénario.\n\nLa grille vérifie schéma JSON strict, action autorisée, entrée canonique, justification\npréautorisée liée aux valeurs, informations manquantes, cohérence risque/action.\nLes tests exercent des erreurs de contrat et un faux appel LLM explicitement simulé,\nsans les enregistrer comme décisions réelles. Aucun score accuracy LLM ou efficacité\nde contrôle n'est revendiqué. Les sources réelles des décisions sont sauvegardées.\nLes notebooks 01–07, les données sources et les métriques historiques sont préservés.\n"
save_results(scenarios, inputs, outputs, validations, methodology)


Résultats validés et sauvegardés : C:\Users\User\GoFlow\results\decision_agent


## 10. Architecture et interprétation

```text
PEMS08
  ↓
XGBoost (prévisions déjà sauvegardées)
  ↓
Prévision +15 min
  ↓
State Builder
  ├── état courant
  ├── tendances strictement passées
  └── voisins directs
  ↓
Decision Agent
  ├── LLM local si disponible et conforme
  └── fallback rule-based
  ↓
Recommandation structurée
  ↓
Orchestrator futur (aucune commande de feux envoyée)
```

Une hausse prévue motive une vigilance accrue ; une baisse prévue n'annule pas
une occupation courante élevée. Des voisins congestionnés motivent une coordination,
sous l'hypothèse de mapping documentée. Les exemples montrent le fonctionnement
des règles et des garde-fous, sans prouver leur efficacité sur un réseau réel.

Les futurs Incident Detection Agent et Emergency Agent fourniront les informations
actuellement inconnues ; Cooperative Traffic Agent fournira le contexte voisin,
Traffic Prediction Agent les prévisions et Orchestrator combinera les informations.
Le présent notebook implémente uniquement Traffic Prediction & Adaptive Signal Control.


In [13]:
# Résumer seulement ce qui a réellement été exécuté.
print('Sources des cinq décisions :', validations['source'].value_counts().to_dict())
print('LLM sélectionné :', ENVIRONMENT['selected_llm'])
print('Aucun réentraînement et aucune modification des métriques historiques.')
if ENVIRONMENT['selected_llm'] is None:
    print('Agent déterministe opérationnel ; interface LLM disponible pour une intégration ultérieure.')


Sources des cinq décisions : {'rule_based': 5}
LLM sélectionné : None
Aucun réentraînement et aucune modification des métriques historiques.
Agent déterministe opérationnel ; interface LLM disponible pour une intégration ultérieure.
